# 🧠 LSTM para Predição de RUL — CMAPSS FD001

## Contexto

Os modelos tabulares do notebook anterior trataram cada ciclo como uma observação independente — dependendo das rolling features para capturar contexto temporal de forma estática. Um Random Forest ou XGBoost não tem memória: ele vê o valor do sensor no ciclo 200 sem saber o que aconteceu nos ciclos 170 a 199.

O LSTM (Long Short-Term Memory) aborda o problema de forma fundamentalmente diferente. Em vez de receber features estáticas, ele recebe uma **sequência de 30 ciclos** e aprende diretamente os padrões temporais — acelerações, tendências, mudanças de regime — sem que essas informações precisem ser explicitamente engenheiradas.

Isso é especialmente relevante para degradação de equipamentos, onde o *ritmo* de mudança dos sensores muitas vezes carrega mais informação do que o valor absoluto em um instante específico.

## O que este notebook implementa

**1. Arquitetura LSTM**
Duas camadas LSTM empilhadas com Dropout para regularização, seguidas de camadas densas para regressão do RUL.

**2. Treinamento com callbacks**
Early Stopping para evitar overfitting e monitoramento das curvas de loss para diagnóstico do treinamento.

**3. Avaliação**
RMSE e S-score no conjunto de teste — mesmas métricas dos modelos tabulares para comparação direta no notebook final.

## 0. Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.dpi'] = 100
plt.rcParams['font.family'] = 'sans-serif'

# S-score — métrica assimétrica da literatura CMAPSS
# Penaliza predições tardias (d > 0) mais severamente que predições antecipadas (d < 0)
def s_score(y_true, y_pred):
    d = y_pred - y_true
    return np.sum(np.where(d < 0, np.exp(-d/13) - 1, np.exp(d/10) - 1))

## 1. Carregamento dos Dados Sequenciais

In [ ]:
X_train = np.load('data/processed/X_train_seq.npy')
y_train = np.load('data/processed/y_train_seq.npy')
X_test  = np.load('data/processed/X_test_seq.npy')
y_test  = np.load('data/processed/y_test_seq.npy')

print(f'X_train: {X_train.shape} → (amostras, timesteps, features)')
print(f'y_train: {y_train.shape}')
print(f'X_test:  {X_test.shape}')
print(f'y_test:  {y_test.shape}')

## 2. Arquitetura da Rede

In [ ]:
# Duas camadas LSTM empilhadas: a primeira retorna a sequência completa
# (return_sequences=True) para que a segunda também processe passo a passo;
# a segunda retorna apenas o estado final, que resume toda a janela de 30 ciclos.
# Dropout(0.2) desliga 20% dos neurônios aleatoriamente durante o treino,
# evitando que a rede memorize padrões específicos do conjunto de treino.
model = Sequential([
    LSTM(64, return_sequences=True, input_shape=(X_train.shape[1], X_train.shape[2])),
    Dropout(0.2),
    LSTM(32, return_sequences=False),
    Dropout(0.2),
    Dense(16, activation='relu'),
    Dense(1)
])

model.compile(optimizer='adam', loss='mse', metrics=['mae'])
model.summary()

## 3. Treinamento

In [ ]:
# EarlyStopping: interrompe o treino se o val_loss não melhorar por 10 épocas
# consecutivas, restaurando os melhores pesos encontrados — evita overfitting.
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True
)

# ReduceLROnPlateau: reduz a taxa de aprendizado pela metade se o modelo
# estagnar por 5 épocas, ajudando a refinar o ajuste final.
reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=5,
    min_lr=1e-6
)

history = model.fit(
    X_train, y_train,
    validation_split=0.15,
    epochs=100,
    batch_size=64,
    callbacks=[early_stop, reduce_lr],
    verbose=1
)

## 4. Histórico de Treinamento

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].plot(history.history['loss'], label='Treino', color='steelblue', linewidth=2)
axes[0].plot(history.history['val_loss'], label='Validação', color='darkred', linewidth=2)
axes[0].set_title('Curva de Loss (MSE)', fontweight='bold')
axes[0].set_xlabel('Época')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].grid(True, linestyle='--', alpha=0.3)

axes[1].plot(history.history['mae'], label='Treino', color='steelblue', linewidth=2)
axes[1].plot(history.history['val_mae'], label='Validação', color='darkred', linewidth=2)
axes[1].set_title('Curva de MAE', fontweight='bold')
axes[1].set_xlabel('Época')
axes[1].set_ylabel('MAE')
axes[1].legend()
axes[1].grid(True, linestyle='--', alpha=0.3)

fig.suptitle('Histórico de Treinamento — LSTM', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('midia/lstm_training_history.png', dpi=150, bbox_inches='tight')
plt.show()

![](midia/deep-treinamento.png)

### Observações 

As curvas de treino e validação evoluem praticamente coladas ao longo de todo o treinamento, sem o distanciamento característico de overfitting. A convergência ocorre rapidamente: a maior parte da queda de loss acontece nas primeiras 8 épocas, com refinamento gradual até o EarlyStopping interromper o treino por estagnação do val_loss.

O MAE de validação estabiliza próximo de 9–10 ciclos de erro médio, patamar competitivo mesmo antes de calcular as métricas formais no conjunto de teste.

**Próximo passo → Seção 5: Avaliação no Conjunto de Teste**

## 5. Avaliação no Conjunto de Teste

In [ ]:
y_pred = model.predict(X_test).flatten()
y_pred = np.clip(y_pred, 0, 125)

rmse_lstm = np.sqrt(mean_squared_error(y_test, y_pred))
ss_lstm   = s_score(y_test, y_pred)

print('=== LSTM ===')
print(f'RMSE:    {rmse_lstm:.2f}')
print(f'S-score: {ss_lstm:.2f}')

### Observações 

O LSTM alcançou RMSE de 14.06 e S-score de 332 — superando significativamente o melhor modelo tabular do notebook anterior (XGBoost: RMSE 17.78, S-score 766). A melhoria é de mais de 21% no RMSE e mais de 55% no S-score.

Essa diferença reflete a vantagem estrutural do LSTM: ele recebe a sequência bruta de 30 ciclos e aprende diretamente os padrões temporais, sem depender de rolling features pré-calculadas como aproximação estática do comportamento sequencial.

**Próximo passo → Seção 6: Predito vs Real**

## 6. Predito vs Real

In [ ]:
plt.figure(figsize=(7, 6))
plt.scatter(y_test, y_pred, alpha=0.6, color='steelblue', s=30)
plt.plot([0, 125], [0, 125], 'r--', linewidth=1.5, label='Ideal')
plt.xlabel('RUL Real')
plt.ylabel('RUL Previsto')
plt.title(f'LSTM — Predito vs Real\nRMSE = {rmse_lstm:.2f}', fontweight='bold')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.3)
plt.tight_layout()
plt.savefig('midia/lstm_pred_vs_real.png', dpi=150, bbox_inches='tight')
plt.show()

![](midia/lstm_pred_vs_real.png)

### Observações 

Os pontos seguem a linha ideal de forma consistente nas regiões de RUL baixo e alto, com maior dispersão na faixa intermediária (80–100 ciclos) — padrão semelhante ao observado nos modelos tabulares, mas com magnitude de erro visivelmente menor.

Um outlier isolado aparece próximo de RUL real = 85, com previsão de ~49 ciclos — caso pontual que não compromete a conclusão geral, mas reforça que mesmo o melhor modelo carrega incerteza residual em casos específicos.

**Próximo passo → Seção 7: Distribuição dos Erros**

## 7. Distribuição dos Erros

In [ ]:
erros = y_pred - y_test

plt.figure(figsize=(8, 5))
plt.hist(erros, bins=25, color='steelblue', alpha=0.7, edgecolor='white')
plt.axvline(0, color='darkred', linewidth=1.5, linestyle='--')
plt.axvline(erros.mean(), color='orange', linewidth=1.5,
            linestyle='--', label=f'Média: {erros.mean():.1f}')
plt.title('LSTM — Distribuição dos Erros', fontweight='bold', fontsize=12)
plt.xlabel('Erro (Previsto − Real)')
plt.ylabel('Contagem')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.3)
plt.tight_layout()
plt.savefig('midia/lstm_error_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

![](midia/lstm_error_distribution.png)

### Observações

A distribuição é quase simétrica e centrada próxima de zero (média 2.6) — bem mais estreita e equilibrada que a observada nos modelos tabulares, que apresentavam viés sistemático de superestimação do RUL. Isso confirma visualmente o ganho de performance refletido no RMSE e no S-score.


## 8. Exportação de Resultados — Comparação Final

In [ ]:
# Exportando predições e ground truth para uso no notebook de comparação final,
# que reúne os resultados de todos os modelos (Regressão Linear, XGBoost, LSTM)
# em uma análise conjunta.
np.save('data/processed/y_pred_lstm.npy', y_pred)
np.save('data/processed/y_test_seq_final.npy', y_test)


## Conclusão 

O LSTM superou significativamente os modelos tabulares do notebook anterior, alcançando RMSE de 14.06 e S-score de 332 — uma melhoria de mais de 21% no RMSE e mais de 55% no S-score em relação ao XGBoost (RMSE 17.78, S-score 766).

A diferença fundamental está em como cada abordagem trata o tempo. Os modelos tabulares dependem de rolling features pré-calculadas (média e desvio padrão móveis) para aproximar o comportamento sequencial — uma representação estática e necessariamente simplificada. O LSTM, ao receber diretamente a sequência bruta de 30 ciclos, aprende sozinho quais padrões temporais — aceleração, mudança de regime, ritmo de variação — são relevantes para a predição, sem que isso precise ser explicitado via engenharia de features.

As curvas de loss mostraram convergência estável e sem overfitting perceptível, com treino e validação evoluindo juntos até a estabilização. A distribuição dos erros confirma esse resultado — quase simétrica, centrada próxima de zero, com cauda muito mais curta que os modelos tabulares.

**Próximo notebook → Comparação Final: Confiabilidade Estatística vs Machine Learning**

O notebook final conecta as duas abordagens estatísticas do projeto — Survival Analysis (Kaplan-Meier, Weibull) e Machine Learning preditivo (XGBoost, LSTM) — sob a ótica de quando cada uma é mais útil na prática de manutenção industrial: planejamento estratégico de longo prazo versus decisão operacional ciclo a ciclo.